# MTM YOLOv8l 학습 (Google Colab)

준비물: 구글 드라이브 `MyDrive/MTM/`에 아래 두 가지를 올려 둔다.
- `dataset.zip` : 사고 영상 변환 결과 (`convert_aihub.py --zip`)
- 교차로 신호 데이터는 드라이브의 원본 압축 파일을 4-0 셀에서 바로 읽는다
- `detector/` 폴더 : 이 저장소의 `detector` 폴더 (train.py 포함)

런타임 > 런타임 유형 변경 > **A100 GPU** 또는 **L4 GPU** 선택 후 위에서부터 차례로 실행한다.

## 1. GPU 확인
`Tesla T4` / `L4` / `A100`이 보이고 마지막 줄이 `True`여야 한다.

In [ ]:
!nvidia-smi
import torch
print(torch.cuda.is_available())

## 2. 구글 드라이브 연결
로그인 창이 뜨면 허용한다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/MTM'
!ls $DRIVE

## 3. 라이브러리 설치

In [ ]:
!pip install -q ultralytics
import ultralytics; ultralytics.checks()

## 4. 데이터를 Colab 내부 디스크로 복사
드라이브에서 바로 읽으면 학습이 매우 느려진다. 압축 파일만 복사해서 내부 디스크에 푼다.

In [ ]:
!cp $DRIVE/dataset.zip /content/
!unzip -q -o /content/dataset.zip -d /content/dataset
!ls /content/dataset/images/train | head -3
!echo "train 이미지 수:" && ls /content/dataset/images/train | wc -l
!echo "train 라벨 수:" && ls /content/dataset/labels/train | wc -l

## 4-0. 교차로 신호 데이터 추가 (드라이브의 압축 파일에서 바로)
TS_4(53GB)를 풀지 않고, 라벨로 고른 사진만 꺼내 `/content/dataset`에 더한다. 카메라 단위 분할, 3초 간격, 교차로·악천후·시간대 고르게.
AI Hub 파일은 확장자가 .zip이지만 **실제로는 7z**라서, 필요한 사진만 꺼내더라도 압축 전체를 한 번 읽는다. **30분~1시간** 걸릴 수 있다.
처음에는 `--dry-run`으로 숫자만 확인(몇 분)하고, 괜찮으면 지우고 다시 실행.

In [ ]:
!apt-get -qq install -y p7zip-full > /dev/null   # AI Hub 파일은 확장자만 zip이고 실제는 7z
CROSS = '/content/drive/MyDrive/MTM/데이터셋/094.교차로 신호 체계, 보행자, 차량 이동 복합 데이터/01.데이터/1.Training'
!ls "{CROSS}/라벨링데이터" "{CROSS}/원천데이터"
%cd $DRIVE/detector
!python convert_cvat.py --labels "{CROSS}/라벨링데이터/TL_4.바운딩박스.zip" --images "{CROSS}/원천데이터/TS_4.바운딩박스.zip" \
    --out /content/dataset --max-images 6000 --dry-run

## 4-1. 빠진 보행자 라벨 채우기 (교차로 신호 데이터를 넣었을 때만)
교차로 신호 데이터(파일 이름이 `x_`로 시작)에는 보행자 라벨이 없어서, GPU로 사전학습 모델을 돌려 채운다. 수천 장 기준 몇 분.

In [ ]:
%cd $DRIVE/detector
!python pseudo_label.py --data /content/dataset --prefix x_ --model yolov8l.pt --device 0
!cat /content/dataset/report_pseudo.json

## 5. data.yaml 확인
`convert_aihub.py`가 데이터와 함께 만든 설정 파일을 그대로 쓴다.

In [ ]:
!cp /content/dataset/data.yaml /content/data.yaml
!cat /content/data.yaml

## 6. 학습 시작
- 결과는 **드라이브**(`MTM/runs`)에 저장되므로 런타임이 끊겨도 남는다.
- 5에폭마다 가중치를 따로 저장한다.
- `CUDA out of memory`가 나면 `--batch 8` → `--batch 4`로 줄인다.

In [ ]:
%cd $DRIVE/detector
!python train.py --data /content/data.yaml --model yolov8l.pt --epochs 100 --imgsz 960 --batch 16 \
    --project $DRIVE/runs --name v8l_cctv_v1

## 7. (끊겼을 때) 이어서 학습
런타임이 끊기면 **1~5번을 다시 실행**한 뒤 이 셀만 실행한다. 처음부터 다시 돌리지 않는다.

In [ ]:
%cd $DRIVE/detector
!python train.py --resume $DRIVE/runs/v8l_cctv_v1/weights/last.pt

## 8. 결과 확인
학습 곡선과 평가 점수. `test_metrics.json`은 학습이 끝까지 돌았을 때 생긴다.

In [ ]:
from IPython.display import Image, display
RUN = f'{DRIVE}/runs/v8l_cctv_v1'
display(Image(f'{RUN}/results.png', width=900))
display(Image(f'{RUN}/confusion_matrix_normalized.png', width=600))
!cat $RUN/test_metrics.json

## 9. 마무리
- 학습된 모델: 드라이브 `MTM/runs/v8l_cctv_v1/weights/best.pt` → 내 PC로 내려받기
- **런타임 > 런타임 연결 해제 및 삭제**를 꼭 누른다. 켜 두면 아무것도 안 해도 컴퓨트 유닛이 계속 줄어든다.